# Merging of the merged Bookings and Measurements dataset and the Materials dataset in respect to the overlapping timeframes in the column created_at

---

In [1]:
import datetime
import os

import duckdb

In [2]:
# Define paths
BASE_PATH_MERGE = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged"
BASE_PATH_FILTERED = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\filtered"
OUTPUT_FILE = os.path.join(BASE_PATH_MERGE, "merged_full_sample_balanced.parquet")

MERGED_FILE = os.path.join(BASE_PATH_MERGE, "merged_bookings_measurements_cleaned.parquet")
FILTERED_MATERIALS_FILE = os.path.join(BASE_PATH_FILTERED, "filtered_materials_encoded_time_cut.parquet")

TIME_WINDOW_DAYS = 70  # Change time frame if desired
SAMPLE_SIZE_PER_CLASS = 5000  # Number of samples per 'book_state'
DROP_CREATED_AT = True  # Set True to drop 'created_at' in the final output

In [3]:
# Connect to DuckDB
con = duckdb.connect()
con.execute("PRAGMA threads=8;")  # Adjust based on CPU cores

In [4]:
# Calculate start_time and end_time
min_created_at_materials = con.execute(f"SELECT MIN(created_at) FROM '{FILTERED_MATERIALS_FILE}'").fetchone()[0]
min_created_at_merged = con.execute(f"SELECT MIN(created_at) FROM '{MERGED_FILE}'").fetchone()[0]
start_time = max(min_created_at_materials, min_created_at_merged)
end_time = start_time + datetime.timedelta(days=TIME_WINDOW_DAYS)

print(f"Time range filter: {start_time} -> {end_time}")

Time range filter: 2025-03-01 02:21:38.306000+01:00 -> 2025-05-10 02:21:38.306000+01:00


In [5]:
# Pre-filter merged and materials datasets by time window
con.execute(f"""
    CREATE TEMP TABLE merged_filtered AS
    SELECT *
    FROM '{MERGED_FILE}'
    WHERE created_at BETWEEN '{start_time}' AND '{end_time}';
""")

con.execute(f"""
    CREATE TEMP TABLE materials_filtered AS
    SELECT *
    FROM '{FILTERED_MATERIALS_FILE}'
    WHERE created_at BETWEEN '{start_time}' AND '{end_time}';
""")

In [6]:
def print_book_state_counts(table_name):
    print(f"\nCounts of book_state in {table_name}:")
    print(con.execute(f"""
        SELECT book_state, COUNT(*) AS count
        FROM {table_name}
        GROUP BY book_state
        ORDER BY book_state;
    """).df())

In [7]:
print_book_state_counts("merged_filtered")
print_book_state_counts("materials_filtered")


Counts of book_state in merged_filtered:
   book_state     count
0           0  41672536
1           1    203965

Counts of book_state in materials_filtered:
   book_state     count
0           0  10901297
1           1       463


In [8]:
# Create a temp table of serial_numbers that appear in materials_filtered
con.execute("""
    CREATE TEMP TABLE materials_serials AS
    SELECT DISTINCT serial_number_id FROM materials_filtered;
""")

In [9]:
# Filter merged_filtered so that only serial_numbers present in materials_serials remain
con.execute("""
            CREATE
            TEMP TABLE merged_filtered_with_materials AS
            SELECT *
            FROM merged_filtered
            WHERE serial_number_id IN (SELECT serial_number_id FROM materials_serials);
            """)
print_book_state_counts("merged_filtered_with_materials")


Counts of book_state in merged_filtered_with_materials:
   book_state    count
0           0  1388687
1           1      222


In [10]:
# Balanced sampling from merged_filtered_with_materials
con.execute(f"""
    CREATE TEMP TABLE merged_sample_balanced AS
    (
        SELECT * FROM merged_filtered_with_materials WHERE book_state = 0 ORDER BY RANDOM() LIMIT {SAMPLE_SIZE_PER_CLASS}
    )
    UNION ALL
    (
        SELECT * FROM merged_filtered_with_materials WHERE book_state = 1 ORDER BY RANDOM() LIMIT {SAMPLE_SIZE_PER_CLASS}
    );
""")
print_book_state_counts("merged_sample_balanced")


Counts of book_state in merged_sample_balanced:
   book_state  count
0           0   5000
1           1    222


In [11]:
# Further filter materials_filtered to only matching serial_number_id in merged_sample_balanced
con.execute("""
            CREATE
            TEMP TABLE materials_reduced AS
            SELECT *
            FROM materials_filtered
            WHERE serial_number_id IN (SELECT DISTINCT serial_number_id FROM merged_sample_balanced);
            """)

print_book_state_counts("materials_reduced")


Counts of book_state in materials_reduced:
   book_state    count
0           0  1478420
1           1        4


In [12]:
# Optional: Count distinct serial_number_id and materials rows matching
count_serials = con.execute("""
                            SELECT COUNT(DISTINCT serial_number_id)
                            FROM merged_sample_balanced;
                            """).fetchone()[0]

count_materials = con.execute(f"""
    SELECT COUNT(*) FROM materials_filtered
    WHERE serial_number_id IN (SELECT DISTINCT serial_number_id FROM merged_sample_balanced);
""").fetchone()[0]

print(f"\nDistinct serial_number_id count in merged_sample_balanced: {count_serials}")
print(f"Count of rows in materials_filtered matching serial_number_id in merged_sample_balanced: {count_materials}")


Distinct serial_number_id count in merged_sample_balanced: 4640
Count of rows in materials_filtered matching serial_number_id in merged_sample_balanced: 1478424


In [ ]:
final_select = f"""
    SELECT
        merged.measure_step_number,
        merged.measure_value,
        merged.book_state,
        merged.measurement_name_encoded,
        merged.measurement_unit_encoded,
        merged.is_within_limits,
        merged.workstep_number_mes,
        merged.book_stamp,
        merged.part_group,
        merged.serial_number_id,
        merged.station_id,
        materials.component_position,
        materials.component_id,
        materials.panel_position,
        materials.supplier_id,
        materials.mounting_place,
        materials.container_number_freq
        {', merged.created_at' if not DROP_CREATED_AT else ''}
    FROM merged_sample_balanced merged
    LEFT JOIN materials_reduced materials
      ON merged.serial_number_id = materials.serial_number_id
     AND ABS(DATE_DIFF('seconds', merged.created_at, materials.created_at)) <= 5
    ORDER BY merged.created_at
"""

Executing join + direct parquet write in DuckDB...


In [ ]:
# Save result to Parquet with compression
con.execute(f"""
    COPY ({final_select})
    TO '{OUTPUT_FILE}' (FORMAT PARQUET, COMPRESSION 'zstd');
""")

print(f"Balanced merged dataset saved to: {OUTPUT_FILE}")
con.close()

In [9]:
# Reconnect to DuckDB
con = duckdb.connect()
# Query distinct book_states and their counts
bookstate_query = f"""
SELECT book_state, COUNT(*) AS count
FROM '{OUTPUT_FILE}'
GROUP BY book_state
ORDER BY count DESC;
"""

bookstate_df = con.execute(bookstate_query).fetchdf()
print(bookstate_df)

con.close()

   book_state      count
0           0  269942247


## Load and inspect merged dataset

In [8]:
con = duckdb.connect()

# View first few rows
print(con.execute(f"SELECT * FROM '{OUTPUT_FILE}' LIMIT 5").fetchdf())

# Count number of rows
row_count = con.execute(f"SELECT COUNT(*) FROM '{OUTPUT_FILE}'").fetchone()[0]
print(f"📊 Row count: {row_count}")

# Show column names and types
print(con.execute(f"DESCRIBE SELECT * FROM '{OUTPUT_FILE}'").fetchdf())

# Quick summary stats
print(con.execute(f"""
    SELECT
        COUNT(*) AS n_rows,
        MIN(book_stamp) AS min_time,
        MAX(book_stamp) AS max_time
    FROM '{OUTPUT_FILE}'
""").fetchdf())
con.close()

   measure_step_number  measure_value  book_state  measurement_name_encoded  \
0                   36          255.0           0                     81322   
1                   36          256.0           0                     81322   
2                   36          255.0           0                     81322   
3                   36          256.0           0                     81322   
4                   36          256.0           0                     81322   

   measurement_unit_encoded  is_within_limits  book_state_1  \
0                   1056712                 1             0   
1                   1056712                 1             0   
2                   1056712                 1             0   
3                   1056712                 1             0   
4                   1056712                 1             0   

   workstep_number_mes                       book_stamp part_group  \
0                    5 2025-03-04 00:35:11.845000+01:00   7a78616d   
1     

## NULL VALUES CHECK

In [9]:
con = duckdb.connect()

# Get all columns
columns = [col[0] for col in con.execute(f"DESCRIBE SELECT * FROM '{OUTPUT_FILE}'").fetchall()]
print("Columns in parquet:")
print(columns)

# Check null counts per column
print("\nNull values per column:")
for col in columns:
    query = f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT({col}) AS non_nulls,
        COUNT(*) - COUNT({col}) AS nulls
    FROM '{OUTPUT_FILE}'
    """
    result = con.execute(query).fetchdf()
    print(f"{col}: nulls = {result['nulls'][0]} / {result['total_rows'][0]} total")

con.close()

Columns in parquet:
['measure_step_number', 'measure_value', 'book_state', 'measurement_name_encoded', 'measurement_unit_encoded', 'is_within_limits', 'book_state_1', 'workstep_number_mes', 'book_stamp', 'part_group', 'serial_number_id', 'station_id', 'component_position', 'component_id', 'panel_position', 'supplier_id', 'mounting_place', 'container_number_freq']

Null values per column:
measure_step_number: nulls = 0 / 269942247 total
measure_value: nulls = 0 / 269942247 total
book_state: nulls = 0 / 269942247 total
measurement_name_encoded: nulls = 0 / 269942247 total
measurement_unit_encoded: nulls = 0 / 269942247 total
is_within_limits: nulls = 0 / 269942247 total
book_state_1: nulls = 0 / 269942247 total
workstep_number_mes: nulls = 0 / 269942247 total
book_stamp: nulls = 0 / 269942247 total
part_group: nulls = 0 / 269942247 total
serial_number_id: nulls = 0 / 269942247 total
station_id: nulls = 0 / 269942247 total
component_position: nulls = 0 / 269942247 total
component_id: null